# IMU Sensor

← [f_h Functions](../f_h_functions.ipynb)

Sensor model, argument reference, and ROSNode wiring for the `imu` block.

## Overview

The IMU block is a **stateless** `DynamicalSystem` (no `f`; only `h`).
`h` extracts orientation, angular velocity, and linear acceleration from the state vector
and adds Gaussian measurement noise, producing an `ndarray(10,)` whose layout
matches `py2ros_imu` exactly.

In a `ROSNode`, the output of `h` is passed directly to `py2ros_imu`,
which converts it to a `sensor_msgs/Imu` message for publication.

## Imu Array Layout

`py2ros_imu` and `ros2py_imu` use the convention:

| Index | Field | Units |
|-------|-------|-------|
| 0 | orientation x (qx) | — |
| 1 | orientation y (qy) | — |
| 2 | orientation z (qz) | — |
| 3 | orientation w (qw) | — |
| 4 | angular velocity x (ωx) | rad/s |
| 5 | angular velocity y (ωy) | rad/s |
| 6 | angular velocity z (ωz) | rad/s |
| 7 | linear acceleration x (ax) | m/s² |
| 8 | linear acceleration y (ay) | m/s² |
| 9 | linear acceleration z (az) | m/s² |

The `imu_h` output must therefore be a `(10,)` array in this order.

## Noise Model

IMU measurement noise is modelled as zero-mean Gaussian:

$$y_k = \begin{bmatrix} q_x \\ q_y \\ q_z \\ q_w \\ \omega_x \\ \omega_y \\ \omega_z \\ a_x \\ a_y \\ a_z \end{bmatrix} + v_k, \qquad v_k \sim \mathcal{N}(0, R_\text{imu})$$

Typical values for a consumer MEMS IMU:

| Component | 1σ accuracy | $R_{ii}$ |
|-----------|------------|----------|
| Orientation (q) | ~0.01 rad | 1×10⁻⁴ |
| Angular velocity | ~0.01 rad/s | 1×10⁻⁴ |
| Linear acceleration | ~0.05 m/s² | 2.5×10⁻³ |

Off-diagonal entries are typically zero (independent axes and sensors).

## Argument Reference

### `xk` — state vector &nbsp; `Float[np.ndarray, "n"]`

Shape `(n,)`. The last 10 elements must be `[qx, qy, qz, qw, wx, wy, wz, ax, ay, az]`.
Extra state components (position, velocity, etc.) at the front are ignored by `imu_h`.

---

### `R_imu` — IMU noise covariance &nbsp; `Float[np.ndarray, "10 10"]`

Shape `(10, 10)`, symmetric positive definite. Encodes IMU measurement uncertainty
across all 10 channels. Diagonal entries are the variances of each channel;
off-diagonal entries model correlations (typically zero).

```python
R_imu = np.diag([
    1e-4, 1e-4, 1e-4, 1e-4,   # orientation quaternion
    1e-4, 1e-4, 1e-4,          # angular velocity (rad/s)
    2.5e-3, 2.5e-3, 2.5e-3,   # linear acceleration (m/s²)
])
```

This value is typically a fixed parameter — pass it as `static_params` in a deployed `ROSNode`.

## Implementation

In [ ]:
import numpy as np
from jaxtyping import Float, jaxtyped
from beartype import beartype
from sensor_msgs.msg import Imu
from dynamicalnodes import DynamicalSystem
from dynamicalnodes.ros2py_py2ros import py2ros_imu, ros2py_imu

In [ ]:
@jaxtyped(typechecker=beartype)
def imu_h(
    xk: Float[np.ndarray, "n"],
    R_imu: Float[np.ndarray, "10 10"],
) -> Float[np.ndarray, "10"]:
    """IMU sensor model; returns [qx, qy, qz, qw, wx, wy, wz, ax, ay, az] with noise.

    Args:
        xk:    State vector (n,) — last 10 elements are [qx, qy, qz, qw, wx, wy, wz, ax, ay, az].
        R_imu: IMU measurement noise covariance (10, 10).
    """
    noise = np.random.multivariate_normal(np.zeros(10), R_imu)
    return xk[-10:] + noise


imu = DynamicalSystem(h=imu_h)

## Worked Example

In [ ]:
xk = np.array([
    39.5296, -119.8138, 1374.0,   # lat, lon, alt (prefix state)
    0.0, 0.0, 0.0, 1.0,           # qx, qy, qz, qw (identity quaternion)
    0.0, 0.0, 0.0,                # wx, wy, wz
    0.0, 0.0, 9.81,               # ax, ay, az
])
R_imu = np.diag([
    1e-4, 1e-4, 1e-4, 1e-4,
    1e-4, 1e-4, 1e-4,
    2.5e-3, 2.5e-3, 2.5e-3,
])

yk = imu.step(xk=xk, R_imu=R_imu)
print("IMU [qx, qy, qz, qw, wx, wy, wz, ax, ay, az]:", yk)

# py2ros: ndarray(10,) → Imu
msg = py2ros_imu(yk)
print(f"Imu: qw={msg.orientation.w:.4f}  az={msg.linear_acceleration.z:.4f} m/s²")

## ROSNode Wiring

Subscribe to a state estimate topic, publish a `sensor_msgs/Imu`.
The `h` output is a `(10,)` array — `py2ros_imu` converts it to the ROS message.

In [ ]:
from dynamicalnodes import ROSNode
from std_msgs.msg import Float64MultiArray
from dynamicalnodes.ros2py_py2ros import ros2py_float64_multiarray

imu_node = ROSNode(
    dynamical_system=imu,
    subscribes_to=[{
        "topic": "/state_estimate",
        "msg_type": Float64MultiArray,
        "arg": "xk",
        "ros2py": ros2py_float64_multiarray,
        "stale_after": 0.1,
    }],
    publishes_to=[{
        "topic": "/imu/data",
        "msg_type": Imu,
        "py2ros": py2ros_imu,
    }],
)

# Simulation step
state_msg = Float64MultiArray()
state_msg.data = list(xk)
imu_out = imu_node.step(xk=state_msg, R_imu=R_imu)
print(f"Published Imu: az={imu_out.linear_acceleration.z:.4f} m/s²")

← [f_h Functions](../f_h_functions.ipynb)